In [0]:
%run '../includes/configuration'

In [0]:
%run '../includes/common_functions'

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date","2024-12-23")
v_file_date = dbutils.widgets.get("p_file_date")

### Ingestión del archivo movie_genre.json

##### Paso 1. Leer el archivo

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
movie_genre_schema = StructType(fields=[
    StructField("movieId", IntegerType(), False),
    StructField("genreId", IntegerType(), False)
])

In [0]:
movie_genre_df = spark.read \
                .schema(movie_genre_schema) \
                .json(f"{bronze_folder_path}/{v_file_date}/movie_genre.json")

##### Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import col, concat, current_timestamp, lit, trim

In [0]:
movie_genre_final_df = movie_genre_df \
                        .withColumnRenamed("movieId","movie_id") \
                        .withColumnRenamed("genreId","genre_id") \
                        .withColumn("ingestion_date", current_timestamp()) \
                        .withColumn("environment", lit(v_environment)) \
                        .withColumn("file_date", lit(v_file_date))

#####Paso 3 - Escribir en parquet y particionar por movie_id

In [0]:
#overwrite_partition("movie_silver","movies_genres","file_date",v_file_date)

In [0]:
merge_condition = "tgt.movie_id = src.movie_id AND tgt.genre_id = src.genre_id AND tgt.file_date = src.file_date"
merge_delta_lake(movie_genre_final_df,"movie_silver","movies_genres",merge_condition,"file_date")

In [0]:
%sql
SELECT file_date, COUNT(1) FROM movie_silver.movies_genres
GROUP BY file_date

file_date,count(1)
2024-12-16,7000
2024-12-23,3000
2024-12-30,2160
